# PROCESO DE CALIBRACIÓN DE PIE DE REY


---

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sympy import symbols, latex, expand, sqrt   # modelo simbolico de u_c(E, L)

In [2]:
def eq(md: str, value, *args):
    disp = f"{md}={value}"
    disp += r"\ \mathrm{ " + " ".join(args) + "}" if args else "" 
    display(Markdown(f"$$ {disp}$$")) 

## 0. DATOS EXPERIMENTALES Y REDUCCIÓN

Mediciones de calibración: cinco puntos del alcance, tres posiciones por punto
(superior, central e inferior), todos a la misma temperatura.

Estas celdas calculan las diferencias de indicación y **escriben las tablas del
informe** en `inf/c3-calibration/pie-de-rey/tablas/`, de modo que los números del
PDF provienen siempre de este cuaderno y no se transcriben a mano.


In [3]:
from pathlib import Path

campana = pd.read_csv("data/pie_de_rey_mediciones.csv")
campana


,BU,VP,X1,X2,X3,T
0,50,50.0,50.0,50.00,50.00,21.5
1,100,100.0,100.0,100.05,100.00,21.5
2,"50, 100",150.0,150.0,150.05,150.05,21.5
3,"25, 75, 100",200.0,200.0,200.05,200.05,21.5
4,"25, 50, 75, 100",250.0,250.0,250.00,250.00,21.5


In [4]:
TAB_DIR = Path("inf/c3-calibration/pie-de-rey/tablas")
TAB_DIR.mkdir(parents=True, exist_ok=True)


def fmt(v, dec=0):
    v = 0.0 if abs(v) < 0.5 * 10 ** (-dec) else v
    return f"{v:.{dec}f}"


def export_table(path, header, rows, caption, label, colspec):
    body = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
        " & ".join(header) + r" \\",
        r"\midrule",
    ]
    body += [" & ".join(row) + r" \\" for row in rows]
    body += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]
    Path(path).write_text("\n".join(body), encoding="utf-8")
    print(f"escrito: {path}")


In [5]:
VP = campana["VP"].to_numpy()
X = campana[["X1", "X2", "X3"]].to_numpy()

# NOTACION: sigma (incertidumbre tipica) equivale a u en la GUM.
# Las correcciones de certificado c_cert se aplican en la celda del valor
# patron corregido (VPC); aqui se trabaja con las diferencias crudas.

D = (X - VP[:, None]) * 1e3        # um, diferencias de indicacion D_kj
E_k = D.max(axis=1) - D.min(axis=1)  # um, rango de las tres posiciones
E_contacto = E_k.max()

eq(r"E", f"{E_contacto:.0f}", r"\mu m")
eq(r"\sigma(L_{i_E})", f"{E_contacto / (2 * np.sqrt(3)):.4f}", r"\mu m")


$$ E=50\ \mathrm{ \mu m}$$

$$ \sigma(L_{i_E})=14.4338\ \mathrm{ \mu m}$$

In [6]:
export_table(
    TAB_DIR / "t1.tex",
    header=[
        r"Bloques patrón (mm)",
        r"$VP$ (mm)",
        r"$X_1$ (mm)",
        r"$X_2$ (mm)",
        r"$X_3$ (mm)",
        r"$T$ ($^\circ$C)",
    ],
    rows=[
        [bu, fmt(vp, 2), fmt(x1, 2), fmt(x2, 2), fmt(x3, 2), fmt(t, 1)]
        for bu, vp, x1, x2, x3, t in zip(
            campana["BU"], VP, X[:, 0], X[:, 1], X[:, 2], campana["T"]
        )
    ],
    caption=(
        "Mediciones de calibración del pie de rey. $X_1$, $X_2$ y $X_3$ son las "
        "lecturas en las posiciones superior, central e inferior de medición de "
        "exteriores."
    ),
    label="tab:pie-rey-datos",
    colspec="lrrrrr",
)


escrito: inf/c3-calibration/pie-de-rey/tablas/t1.tex


## 0.1 CÁLCULO ESTADÍSTICO ELEMENTAL

Antes de introducir correcciones térmicas o de certificado se calcula la estadística directa de las mediciones de la Tabla 1: media por punto, desviación estándar de las tres posiciones y error de indicación sin corregir.

El promedio de las $n = 3$ posiciones se obtiene con la media muestral y su dispersión con la varianza insesgada, ambas deducidas en el capítulo 1:

$$\bk{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_{ki}, \qquad
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}\left(X_{ki} - \bk{X}_n\right)^2,$$

y la incertidumbre de ese promedio sigue la desviación estándar de la media,

$$\sigma(\bk{X}_n) = \frac{s}{\sqrt{n}}.$$

Lo que no tiene precedente es el error de indicación elemental, que se define sin corrección térmica ni de certificado como la diferencia entre el promedio y el valor nominal del patrón:

$$E_k^{\text{simple}} = \bk{X}_n - VP_k.$$


In [7]:
# Estadística elemental por punto de calibración (sin correcciones)
n_pos = X.shape[1]

X_media = X.mean(axis=1)                  # mm
X_s = X.std(axis=1, ddof=1)               # mm, desviación estándar de las 3 posiciones
X_s_media = X_s / np.sqrt(n_pos)          # mm, incertidumbre del promedio
X_rango = (X.max(axis=1) - X.min(axis=1)) * 1e3   # um

estadistica = pd.DataFrame({
    "VP": VP,
    "media": X_media,
    "s": X_s,
    "s_media": X_s_media,
    "rango_um": X_rango,
    "E_simple_um": (X_media - VP) * 1e3,
})

print(estadistica.round(4).to_string(index=False))
print()
print(f"Desviaciones estándar nulas en: {list(VP[X_s == 0])} mm")


   VP    media      s  s_media  rango_um  E_simple_um
 50.0  50.0000 0.0000   0.0000       0.0       0.0000
100.0 100.0167 0.0289   0.0167      50.0      16.6667
150.0 150.0333 0.0289   0.0167      50.0      33.3333
200.0 200.0333 0.0289   0.0167      50.0      33.3333
250.0 250.0000 0.0000   0.0000       0.0       0.0000

Desviaciones estándar nulas en: [np.float64(50.0), np.float64(250.0)] mm


### 0.1.1 INTERPRETACIÓN DE LA DISPERSIÓN

Las desviaciones estándar resultan nulas en $50$ mm y en $250$ mm, y de $0.0289$ mm en los tres puntos restantes. Un valor nulo no indica ausencia de dispersión: indica que las tres lecturas coinciden exactamente dentro de la graduación del instrumento.

La resolución del pie de rey es $d = 0.05$ mm (ecuación~\eqref{eq:resolucion-vernier}), de modo que las lecturas de una misma posición solo pueden diferir en múltiplos exactos de esa cantidad. Con tres repeticiones por punto, la estadística tipo A (GUM, JCGM 100:2008) no puede discriminar dispersiones por debajo del paso de graduación: la única información disponible es que las tres lecturas son iguales o difieren en un paso, lo que arroja $s = 0$ o $s = d/\sqrt{3}$.

Por esa razón el cálculo por punto no caracteriza la repetibilidad del instrumento. La estimación tipo A aprovechable es la serie de cinco lecturas repetidas en un mismo punto, la única del conjunto que proviene de observaciones repetidas, que se presenta a continuación.

### 0.1.2 REPETIBILIDAD DEL INSTRUMENTO

La repetibilidad se evalúa con $n = 5$ lecturas consecutivas en el punto de $250$ mm. El rango y la desviación estándar muestral son

$$R = \max_p R_p - \min_p R_p,
\qquad
s_R^2 = \frac{1}{n-1}\sum_{p=1}^{n}\left(R_p - \bk{R}_n\right)^2,$$

y la incertidumbre del promedio repetido, conforme a la desviación estándar de la media,

$$\sigma(L_{i_R}) = \frac{s_R}{\sqrt{n}}.$$


In [8]:
# Repetibilidad: única estimación tipo A con n = 5 (mismos nombres que usa la
# sección de componentes; se calculan aquí para el cálculo elemental)
rep = pd.read_csv("data/pie_de_rey_repetibilidad.csv")["medicion"].to_numpy()
n_rep = len(rep)

R_rango = (rep.max() - rep.min()) * 1e3
S_R = rep.std(ddof=1) * 1e3
u_rep = S_R / np.sqrt(n_rep)

print(f"lecturas      = {list(rep)} mm")
print(f"n             = {n_rep}")
print(f"media         = {rep.mean():.3f} mm")
print(f"R             = {R_rango:.0f} um")
print(f"s_R           = {S_R:.2f} um")
print(f"sigma(L_i_R)  = {u_rep:.2f} um")


lecturas      = [np.float64(250.1), np.float64(250.05), np.float64(250.1), np.float64(250.1), np.float64(250.05)] mm
n             = 5
media         = 250.080 mm
R             = 50 um
s_R           = 27.39 um
sigma(L_i_R)  = 12.25 um


In [9]:
export_table(
    TAB_DIR / "t7.tex",
    header=[
        r"$VP$ (mm)",
        r"$\bk{X}_n$ (mm)",
        r"$s$ (mm)",
        r"$s/\sqrt{3}$ (mm)",
        r"$E^{\text{simple}}$ ($\mu$m)",
    ],
    rows=[
        [fmt(vp, 0), fmt(m, 4), fmt(s, 4), fmt(sm, 4), fmt(e, 1)]
        for vp, m, s, sm, e in zip(
            VP, X_media, X_s, X_s_media, (X_media - VP) * 1e3
        )
    ],
    caption=(
        "Cálculo estadístico elemental por punto de calibración. $\\bk{X}_n$ es la "
        "media de las tres posiciones, $s$ la desviación estándar muestral y "
        "$E^{\\text{simple}}$ el error de indicación sin correcciones."
    ),
    label="tab:pie-rey-estadistica",
    colspec="rrrrr",
)


escrito: inf/c3-calibration/pie-de-rey/tablas/t7.tex


La Tabla~\ref{tab:pie-rey-estadistica} recoge el cálculo elemental. Los errores simples llegan a $33.3\ \mu\text{m}$ en los puntos de $150$ mm y $200$ mm y se anulan en los extremos del alcance, donde las tres lecturas coinciden.

Este cálculo, sin embargo, no distingue el origen de esas diferencias. El error de indicación medido así mezcla el comportamiento del instrumento con el estado del patrón y con las condiciones ambientales en que se efectuó la medición: un valor no nulo puede deberse al pie de rey, a la pila de bloques o a la dilatación térmica de ambos cuerpos. Tampoco permite asignar una incertidumbre, porque la dispersión por punto es incompatible con la resolución del instrumento.

El modelo que se desarrolla a continuación descompone el error en sus contribuciones y permite propagar la incertidumbre de cada una hasta el resultado final.


In [10]:
export_table(
    TAB_DIR / "t3.tex",
    header=[
        r"$VP$ (mm)",
        r"$D_1$ ($\mu$m)",
        r"$D_2$ ($\mu$m)",
        r"$D_3$ ($\mu$m)",
        r"$E_k$ ($\mu$m)",
    ],
    rows=[
        [fmt(vp, 0), fmt(d1, 0), fmt(d2, 0), fmt(d3, 0), fmt(ek, 0)]
        for vp, d1, d2, d3, ek in zip(VP, D[:, 0], D[:, 1], D[:, 2], E_k)
    ],
    caption=(
        "Diferencias de indicación por posición y dispersión $E_k$ en cada punto "
        "de calibración."
    ),
    label="tab:pie-rey-diferencias",
    colspec="rrrrr",
)


escrito: inf/c3-calibration/pie-de-rey/tablas/t3.tex


In [11]:
a_steel = 11.5e-6  # 1/C

# Desviaciones de certificado por bloque (mm). No se dispone de la desviacion
# individual de cada bloque, de modo que se adopta c_cert = 0 (hipotesis).
C_CERT_BLOQUE = {25.0: 0.0, 50.0: 0.0, 75.0: 0.0, 100.0: 0.0}


def c_cert(pila: str) -> float:
    return sum(C_CERT_BLOQUE[float(b)] for b in pila.split(","))


c_cert_k = np.array([c_cert(bu) for bu in campana["BU"]])
T = campana["T"].to_numpy()

VPC = VP + c_cert_k + a_steel * VP * (T - 20)     # mm
PC = X.mean(axis=1) * (1 - a_steel * (T - 20))    # mm
err_ind = (PC - VPC) * 1e3                        # um

eq(r"VPC", f"{VPC[-1]:.4f}", r"mm")
eq(r"PC", f"{PC[-1]:.4f}", r"mm")

# --- Incertidumbre del bloque patron ---
# Certificado de calibracion: UL = 2 sqrt(a^2 + b^2 L^2), a = 41.46 nm, b = 0.55 nm/mm, k = 2
A_CAL = 41.46e-9      # nm -> m
B_CAL = 0.55e-9       # nm/mm -> m/mm
K_CAL = 2

# Deriva anual maxima permitida, ISO 3650 grado 1: +/-(0.05 um + 0.5e-6 L)
A_DER = 0.05          # um
B_DER = 0.5e-6        # um/mm

bloques = pd.read_csv("data/bloques_patron_p.csv")["longitud (mm)"].to_numpy()


def u_cal_q(l):
    return 2 * np.sqrt(A_CAL ** 2 + B_CAL ** 2 * l ** 2) / K_CAL * 1e6   # um


def u_der_q(l):
    return (A_DER + B_DER * l) / np.sqrt(3)                              # um


u_BP_cal = np.sqrt(np.sum(u_cal_q(bloques) ** 2))
u_BP_der = np.sqrt(np.sum(u_der_q(bloques) ** 2))
u_BP = np.sqrt(u_BP_cal ** 2 + u_BP_der ** 2)

eq(r"\sigma(L_{BP,cal})", f"{u_BP_cal:.4f}", r"\mu m")
eq(r"\sigma(L_{BP,der})", f"{u_BP_der:.4f}", r"\mu m")
eq(r"\sigma(L_{BP})", f"{u_BP:.4f}", r"\mu m")

$$ VPC=250.0043\ \mathrm{ mm}$$

$$ PC=249.9957\ \mathrm{ mm}$$

$$ \sigma(L_{BP,cal})=0.1120\ \mathrm{ \mu m}$$

$$ \sigma(L_{BP,der})=0.0578\ \mathrm{ \mu m}$$

$$ \sigma(L_{BP})=0.1260\ \mathrm{ \mu m}$$

In [12]:
export_table(
    TAB_DIR / "t2.tex",
    header=[
        r"$VP$ (mm)",
        r"$VPC$ (mm)",
        r"$PC$ (mm)",
        r"Error de indicación ($\mu$m)",
    ],
    rows=[
        [fmt(vp, 2), fmt(vpc, 3), fmt(pc, 3), fmt(err, 1)]
        for vp, vpc, pc, err in zip(VP, VPC, PC, err_ind)
    ],
    caption=(
        "Valor patrón corregido, promedio corregido y error de indicación en cada "
        "punto de calibración."
    ),
    label="tab:pie-rey-vpc",
    colspec="rrrr",
)

escrito: inf/c3-calibration/pie-de-rey/tablas/t2.tex


In [13]:
cil = pd.read_csv("data/pie_de_rey_cilindro.csv")
VP_cil = cil["VP"].to_numpy()
X_cil = cil[["X1", "X2", "X3"]].to_numpy()

D_cil = (X_cil - VP_cil[:, None]) * 1e3  # um

ext = D_cil[0]
S_EI = (D_cil[1] - ext).mean()            # um
S_EP = (D_cil[2] - ext).mean()            # um
S_cambio = max(abs(S_EI), abs(S_EP))

eq(r"S_{E-I}", f"{S_EI:.1f}", r"\mu m")
eq(r"S_{E-P}", f"{S_EP:.1f}", r"\mu m")
eq(r"\sigma(L_{i_S})", f"{S_cambio / (2 * np.sqrt(3)):.2f}", r"\mu m")

$$ S_{E-I}=16.7\ \mathrm{ \mu m}$$

$$ S_{E-P}=33.3\ \mathrm{ \mu m}$$

$$ \sigma(L_{i_S})=9.62\ \mathrm{ \mu m}$$

In [14]:
rep = pd.read_csv("data/pie_de_rey_repetibilidad.csv")["medicion"].to_numpy()
n_rep = len(rep)

R_rango = (rep.max() - rep.min()) * 1e3          # um
S_R = rep.std(ddof=1) * 1e3                      # um
u_LiR = S_R / np.sqrt(n_rep)

eq(r"n", f"{n_rep}")
eq(r"\overline{X}", f"{rep.mean():.3f}", r"\mathrm{mm}")
eq(r"R", f"{R_rango:.0f}", r"\mu m")
eq(r"S(R)", f"{S_R:.2f}", r"\mu m")
eq(r"\sigma(L_{i_R})", f"{u_LiR:.2f}", r"\mu m")

$$ n=5$$

$$ \overline{X}=250.080\ \mathrm{ \mathrm{mm}}$$

$$ R=50\ \mathrm{ \mu m}$$

$$ S(R)=27.39\ \mathrm{ \mu m}$$

$$ \sigma(L_{i_R})=12.25\ \mathrm{ \mu m}$$

In [15]:
export_table(
    TAB_DIR / "t4.tex",
    header=[r"$p$", r"$X_p$ (mm)"],
    rows=[[str(p + 1), fmt(x, 2)] for p, x in enumerate(rep)],
    caption=(
        f"Repetibilidad en el punto de mayor error de contacto ($250$ mm), "
        f"$n = {n_rep}$ mediciones repetidas."
    ),
    label="tab:pie-rey-repetibilidad",
    colspec="rr",
)

escrito: inf/c3-calibration/pie-de-rey/tablas/t4.tex


In [16]:
# Tabla 5: el bloque de indicacion tiene tres columnas (posiciones X1-X3)
# y dentro de cada una se comparan las tres aplicaciones. Asi las filas
# agrupadas son las posiciones de medicion (k) y las columnas la aplicacion.
APLIC = ["Exteriores", "Interiores", "Profundidad"]
POS = ["1 (superior)", "2 (central)", "3 (inferior)"]

export_table(
    TAB_DIR / "t5.tex",
    header=[
        r"Posición $k$",
        r"$X_{k,1}$ (mm)",
        r"$X_{k,2}$ (mm)",
        r"$X_{k,3}$ (mm)",
        r"$D_{k,1}$ ($\mu$m)",
        r"$D_{k,2}$ ($\mu$m)",
        r"$D_{k,3}$ ($\mu$m)",
    ],
    rows=[
        [POS[k]]
        + [fmt(X_cil[j, k], 2) for j in range(3)]
        + [fmt(D_cil[j, k], 0) for j in range(3)]
        for k in range(3)
    ],
    caption=(
        "Cambio de escala sobre el cilindro patrón de $10$ mm. Las columnas "
        r"corresponden a la medición de exteriores, interiores y profundidad, "
        "respectivamente; las filas, a la posición sobre cada superficie."
    ),
    label="tab:pie-rey-cilindro",
    colspec="lrrrrrr",
)

escrito: inf/c3-calibration/pie-de-rey/tablas/t5.tex



## 1. DEBIDO A LA INDICACION DEL PIE DE REY
### 1.1 DEBIDO A ERROR DE CONTACTO DE LA SUPERFICIE PARCIAL

$$
u(L_{i_E}) = \frac{E}{2\sqrt3}
$$

Para el caso de dado $E = 10 \mu m$

Remplazando

In [17]:
E = E_contacto  # um, calculado en la sección 0
eq(r"E", f"{E}", r'\mu m')

$$ E=50.00000000001137\ \mathrm{ \mu m}$$

In [18]:
ULi_E = E / (2 * 3**0.5)
eq(r"u(L_{i_{E}})", f"{ULi_E:.4f}", r'\mu m')

$$ u(L_{i_{E}})=14.4338\ \mathrm{ \mu m}$$

### 1.2 DEBIDO AL ERROR DE REPETIBILIDAD DE CONTACTO DE LA SUPERFICIE
$$
u(L_{i_R}) = \frac{S(\overline Ro)}{\sqrt{n}}
$$

Donde

$$
S(\overline R) = \sqrt {
    \frac{1}{n - 1} \sum_{p = 1}^{n}\left(R_p - \overline R\right)^2
}
$$

para el caso se tiene $n = 5$, con los iguientes datos

In [19]:
n = 5
df = pd.read_csv("data/piederey.csv")
df

,MEDICIONES (mm)
0,250.10
1,250.05
2,250.10
3,250.10
4,250.01


In [20]:
R_mean = np.mean(df)
eq(r"\overline R", f"{R_mean:.3f}")

$$ \overline R=250.072$$

In [21]:
SRmean = np.sqrt(
    1/(n -1) * sum(
        [(x - R_mean)**2 for x in df["MEDICIONES (mm)"]]
    )
)

eq(r"S(\overline R)", f"{SRmean:.4f}")

$$ S(\overline R)=0.0409$$

In [22]:
ULi_R = SRmean * 1000 / np.sqrt(n)

eq(r"u(L_{i_R})", f"{ULi_R:.4f}", r'\mu m')

$$ u(L_{i_R})=18.2757\ \mathrm{ \mu m}$$


### 1.3 DEBIDO AL ERROR POR CAMBIO DE ESCALA
#### DE MEDICION DE EXTERIORES A MEDICION DE INTERIORES

$$
u(L_{i_{S_{E-I}}}) = \frac{S_{E-I}}{2\sqrt 3}
$$

en este caso se tiene

In [23]:
SE_I = 3.5 # um
eq(r"S_{E-I}", f"{SE_I}", r'\mu m')

$$ S_{E-I}=3.5\ \mathrm{ \mu m}$$

In [24]:
ULiSE_I = 3.5/(2 * np.sqrt(3))
eq(r"u(L_{i_{S_{E-I}}})", f"{ULiSE_I:.4f}", r'\mu m')

$$ u(L_{i_{S_{E-I}}})=1.0104\ \mathrm{ \mu m}$$


#### DE MEDICION DE EXTERIORES A MEDICION DE PROFUNDIDAD


$$
u(L_{i_{S_{E-P}}}) = \frac{S_{E-P}}{2\sqrt 3}
$$

Para el cado dado se tiene



In [25]:
SE_P = 10 # um
eq(r"S_{E-P}", f"{SE_P}", r'\mu m')

$$ S_{E-P}=10\ \mathrm{ \mu m}$$

In [26]:
ULiSE_P = 10/(2 * np.sqrt(3))
eq(r"u(L_{i_{S_{E-P}}})", f"{ULiSE_P:.4f}", r'\mu m')

$$ u(L_{i_{S_{E-P}}})=2.8868\ \mathrm{ \mu m}$$

Por lo que el máximo valore es:
$$
u(L_{i_{S}}) = \max\left(u(L_{i_{S_{E-I}}}), u(L_{i_{S_{E-P}}})\right)
$$

In [27]:
ULi_S = np.max([ULiSE_I, ULiSE_P])

eq(r"u(L_{i_{S}})", f"{ULi_S:.4f}", r'\mu m')

$$ u(L_{i_{S}})=2.8868\ \mathrm{ \mu m}$$

### 1.4 DEBIDO AL EROR DE CONTACTO LINEAL

$$
u(L_{i_L}) = \frac{L}{2\sqrt3}
$$

Para el caso dedo se tiene

In [28]:
L = 10 # um
eq(r"L", f"{L}", r'\mu m')

$$ L=10\ \mathrm{ \mu m}$$

In [29]:
ULi_L = L/(2 * np.sqrt(3))
eq(r"u(L_{i_L})", f"{ULi_L:.4f}", r'\mu m')

$$ u(L_{i_L})=2.8868\ \mathrm{ \mu m}$$

### 1.5 DEBIDO AL ERROR DE CONTANCTO DE SUPERFICIE COMPLETA

$$
u(L_{i_J}) = \frac{J}{2\sqrt3}
$$

Para el caso dado se tiene que:

In [30]:
J = 10 # um
eq(r"J", f"{J}", r'\mu m')

$$ J=10\ \mathrm{ \mu m}$$

In [31]:
ULi_J = J/(2 * np.sqrt(3))
eq(r"u(L_{i_J})", f"{ULi_J:.4f}", r'\mu m')

$$ u(L_{i_J})=2.8868\ \mathrm{ \mu m}$$

### 1.6 DEBIDO A LA DISTANCIA DE CRUCE DE LAS SUPERFICIES DE MEDICION PARA MEDICION DE INTERIORES

$$
u(L_{i_K}) = \frac{K}{2\sqrt3}
$$

Para el caso dado se tiene

In [32]:
K = 0 # um
eq(r"K", f"{K}", r'\mu m')

$$ K=0\ \mathrm{ \mu m}$$

In [33]:
ULi_K = K/(2 * np.sqrt(3))
eq(r"u(L_{i_K})", f"{ULi_K:.4f}", r'\mu m')

$$ u(L_{i_K})=0.0000\ \mathrm{ \mu m}$$

### 1.7 DEBIDO A LA RESOLUCION DEL PIE DE REY

$$
u(L_{i_r}) = \frac{r}{2\sqrt3} = \frac{d/m}{\sqrt3}
$$

Para el caso dado se tiene


In [34]:
d = 10 # um
m = 2 # Por ser un pie de rey digital

eq(r"d", f"{d}", r"\mu m \quad", f"{m=}")

$$ d=10\ \mathrm{ \mu m \quad m=2}$$

In [35]:
ULi_r = d/m/(np.sqrt(3))
eq(r"u(L_{i_r})", f"{ULi_r:.4f}", r'\mu m')

$$ u(L_{i_r})=2.8868\ \mathrm{ \mu m}$$

Combinando las contribuciones de las incertidumbres de $u(L_i)$, se tiene
$$
u(L_i) = \sqrt{
    \left(\frac{E}{2\sqrt3}\right)^2 +
    \left(\frac{S(\overline R)}{\sqrt{n}}\right)^2 +
    \left(
        \max\left[
            \frac{S_{E-I}}{2\sqrt3},
            \frac{S_{E-P}}{2\sqrt3}
        \right]
    \right)^2 +
    \left(\frac{L}{2\sqrt3}\right)^2 +
    \left(\frac{J}{2\sqrt3}\right)^2 +
    \left(\frac{K}{2\sqrt3}\right)^2 +
    \left(\frac{d/m}{\sqrt3}\right)^2
}
$$

In [36]:
UL_i = np.sqrt(
    sum([ x**2 for x in [
        ULi_E,
        ULi_R,
        ULi_S,
        ULi_L,
        ULi_J,
        ULi_K,
        ULi_r
    ]
    ])
)

eq(r"u(L_{i})", f"{UL_i:.4f}", r'\mu m')

$$ u(L_{i})=23.9931\ \mathrm{ \mu m}$$

## 2 CONTRIBUCION A LA INCERTIDUMBRE DEVIDO AL BLOQE PATRON

Bloques patron utilizados

In [37]:
df = pd.read_csv("data/bloques_patron.csv")
df

,longitud (mm)
0,30
1,80
2,90
3,100


### 2.1 DEBIDO A LA INCERTIDUMBRE SEGUN CERTIFICADOA DE CALIBRACION DEL BLOQUE PATRON

$$
u(L_{BP_{cal}}) = \frac{UL_{BP_{cal}}}{k}
$$

Considerando que el máximo valor medido es 300 mm, si la incertidumbre según el certificado de calibración es:
$$
UL_{BP_{cal}} = 2\sqrt{41.36² + 0.55² * l²}\ \mathrm{nm};\ \mathrm{con\ k = 2}
$$

utilizando los 4 bloques patron se tiene

In [38]:
ULBP_cal = lambda l: 2 * np.sqrt(41.46 ** 2 + 0.55 ** 2 * l**2) * 1e-9 # nm 
k = 2

UL_BP_cal = np.sqrt(sum([(ULBP_cal(l)/k) ** 2 for l in df.iloc[:,0]])) /1e-6

eq(r"u(L_{BP_{cl}})", f"{UL_BP_cal:.4f}", r"\mu m")

$$ u(L_{BP_{cl}})=0.1207\ \mathrm{ \mu m}$$

### 2.2 DEBIDO  LA DERIVA DEL BLOQUE PATRON

$$
u(L_{BL_{der}}) = \frac{U_{der}}{\sqrt 3}
$$

Considrenado que el máximo valor medio es 300 nm y tomando los datos de la tabla 1 para bloques de grado1 ,y considerando los 4 bloques
se tiene que

In [39]:
U_der = lambda l: 0.05 + 0.5e-6 * l

UL_der = np.sqrt(sum([(U_der(l)/np.sqrt(3)) ** 2 for l in df.iloc[:, 0]]))

eq(r"u(L_{der})", f"{UL_der:.4f}", r"\mu m")

$$ u(L_{der})=0.0578\ \mathrm{ \mu m}$$

La contribución de la incertidumbre del bloque patrón es:
$$
u(L_{BP}) = \sqrt{(u(L_{BP_{cal}})² + (u(L_{BP_{der}}))²}
$$

remplazando los resultados anteriores

In [40]:
UL_BP = np.sqrt(UL_BP_cal**2 + UL_der**2)

eq(r"u(L_{BP})", f"{UL_BP:.4f}", r"\mu m")

$$ u(L_{BP})=0.1338\ \mathrm{ \mu m}$$

### 3. DEBIDO AL COEFICIENTE DE EXPANSION TERMICA DEL BLOQUE
$$
u(\alpha_{BP}) = \frac{\Delta\alpha_{BP}}{\sqrt3}
$$

para el caso se tiene

In [41]:
da_BP = 1 # ppm/c
eq(r"\Delta \alpha_{BP}", f"{da_BP}")

uda_BP = da_BP/1_000_000/np.sqrt(3)

eq(r"u(a_{BP})", f"{uda_BP:.8f}", r"^\circ C^{-1}")

$$ \Delta \alpha_{BP}=1$$

$$ u(a_{BP})=0.00000058\ \mathrm{ ^\circ C^{-1}}$$

### 4. DEBIDO A LAS DIFERENCIAS DE LOS COEFICIENTES DE EXPANSION TERMICA DEL PIE DE REY Y EL BLOQUE
$$
u(\delta \alpha) = \sqrt{
    \left(
        \frac{\Delta\alpha_{BP}}{\sqrt3}
    \right)² +
    \left(
        \frac{\Delta\alpha_{i}}{\sqrt3}
    \right)²
    }
$$

Para el caso dado se tiene que

In [42]:
da_i = 1 # ppm/c

eq(r"\Delta \alpha_{BP}", f"{da_BP}")
eq(r"\Delta \alpha_{i}", f"{da_i}")

uda = np.sqrt(uda_BP**2 + (da_i*1e-6/np.sqrt(3))**2)

eq(r"u(\delta \alpha)", f"{uda:.8f}", r"^\circ C^{-1}")


$$ \Delta \alpha_{BP}=1$$

$$ \Delta \alpha_{i}=1$$

$$ u(\delta \alpha)=0.00000082\ \mathrm{ ^\circ C^{-1}}$$

## 5. DEBIDO A LAS DIFERNCIAS DE TEMPERATURA ENTRE EL AMBIENTE Y EL PIDE DE REY

$$
u(\Delta T_i) = \sqrt{
    \left(\frac{UT_{cert}}{k}\right)² +
    \left(\frac{\Delta t_{Ai}}{\sqrt3}\right)^2 +
    \left(\frac{d_T/m_T}{\sqrt3}\right)^2 +
    \left(\frac{UT_{der}}{\sqrt(3)}\right)^2
}
$$


Para nuestro caso se tiene que:
$
UT_{cert} = 0.5 {}^\circ C;
\Delta t_{Ai} = 2 {}^\circ C;
d_T = 0.2 {}^\circ C;
m_T = 5 {}^\circ C;
k = 2 {}^\circ C;
UT_{der} = 0.5 {}^\circ C
$

In [43]:
# Datos
UT_cert = 0.5
dt_Ai = 2
dT = 0.2
mT = 5
k = 2
UT_der = 0.5

# Calculos
UDTi = np.sqrt((UT_cert/k)**2 + (dt_Ai/np.sqrt(3))**2 + (dT/mT/np.sqrt(3))**2 + (UT_der/np.sqrt(3))**2)

eq(r"u(\Delta T_i)", f"{UDTi:.4f}", r"^\circ C^{-1}")

$$ u(\Delta T_i)=1.2164\ \mathrm{ ^\circ C^{-1}}$$

## 6. DEBIDO A LAS DIFERNCIAS DE TEMPERATURA DE LOS BLOQUES Y EL PIE DE REY

$$u(\delta T) = \sqrt{
    \left(\frac{UT_{cert}}{k}\right)² +
    \left(\frac{\Delta t_{Ai}}{\sqrt3}\right)^2 +
    \left(\frac{d_T/m_T}{\sqrt3}\right)^2 +
    \left(\frac{UT_{der}}{\sqrt(3)}\right)^2
}
$$


Para nuestro caso se tiene que:
$
UT_{cert} = 0.5 {}^\circ C;
\Delta t_{Ai} = 2 {}^\circ C;
d_T = 0.2 {}^\circ C;
m_T = 5 {}^\circ C;
k = 2 {}^\circ C;
UT_{der} = 0.5 {}^\circ C
$

In [44]:
# Datos
UT_cert = 0.5
dt_Ai = 2
dT = 0.2
mT = 5
k = 2
UT_der = 0.5

# Calculos
UdT = np.sqrt((UT_cert/k)**2 + (dt_Ai/np.sqrt(3))**2 + (dT/mT/np.sqrt(3))**2 + (UT_der/np.sqrt(3))**2)

eq(r"u(\delta T)", f"{UdT:.4f}", r"^\circ C^{-1}")

$$ u(\delta T)=1.2164\ \mathrm{ ^\circ C^{-1}}$$

In [45]:
d_res = 0.05   # mm, una division de escala del 530-115
m_res = 2      # vernier
L_lineal = 10.0  # um, error de contacto lineal (cilindro)
J_plano = 0.0    # um
K_cruce = 0.0    # um

u_E = E_contacto / (2 * np.sqrt(3))
u_R = u_LiR
u_S = S_cambio / (2 * np.sqrt(3))
u_L = L_lineal / (2 * np.sqrt(3))
u_J = J_plano / (2 * np.sqrt(3))
u_K = K_cruce / (2 * np.sqrt(3))
u_r = (d_res / m_res) / np.sqrt(3) * 1e3   # um

comp = {
    "Contacto parcial $E$": u_E,
    "Repetibilidad $R$": u_R,
    "Cambio de escala $S$": u_S,
    "Contacto lineal $L$": u_L,
    "Superficie completa $J$": u_J,
    "Cruce de interiores $K$": u_K,
    "Resolución $r$": u_r,
}
u_Li = np.sqrt(sum(v ** 2 for v in comp.values()))
u_cE = np.sqrt(u_Li ** 2 + u_BP ** 2)
U_E = 2 * u_cE

In [46]:
errors = [
    UL_i,
    UL_BP,
    uda,
    UDTi,
    uda_BP,
    UdT
]

## 8. Resultados

In [47]:
# --- Modelo completo de u_c^2(E) como funcion de la longitud L (mm) ---
# Coeficientes de sensibilidad de la ecuacion de propagacion del informe.
# Todas las componentes provienen de las mediciones de este notebook.

da_i = 11.5e-6          # 1/C, pie de rey (acero)
da_BP = 11.5e-6         # 1/C, bloques patron (acero)
d_alpha = da_i - da_BP  # delta alpha = 0 (mismo material)
dT_val = 0.0            # delta T = 0 (misma temperatura bloque/instrumento)
DT_val = 1.5            # Delta T_i = 21.5 - 20 C (temperatura medida)

uda_BP = 1e-6 / np.sqrt(3)          # 1/C
u_dalpha = np.sqrt(2) * 1e-6 / np.sqrt(3)   # 1/C

# Incertidumbre del termometro (sin fluctuacion observada: T constante)
UT_cert, UT_der, dT_res, mT, kT = 0.5, 0.5, 0.2, 5, 2
u_DTi = np.sqrt((UT_cert/kT)**2 + (dT_res/mT/np.sqrt(3))**2 + UT_der**2/3)
u_dT = u_DTi

# Dependencias: u_Li (balance de componentes) y u_BP (bloque patron),
# calculadas en las celdas anteriores de esta seccion.

L = symbols("L")

u2cE_expr = (1 + d_alpha * DT_val) ** 2 * u_Li ** 2
u2cE_expr += (-1 + da_BP * dT_val) ** 2 * u_BP ** 2
u2cE_expr += (L * DT_val) ** 2 * u_dalpha ** 2
u2cE_expr += (L * d_alpha) ** 2 * u_DTi ** 2
u2cE_expr += (L * dT_val) ** 2 * uda_BP ** 2
u2cE_expr += (L * da_BP) ** 2 * u_dT ** 2

u2cE = expand(u2cE_expr)
eq(r"u_c^2(E,L)", latex(u2cE), r"\mu m^2")

ucE_L = sqrt(u2cE)
eq(r"u_c(E,L)", latex(ucE_L), r"\mu m")

UE_L = sqrt(4 * u2cE)
eq(r"U(E,L)", latex(UE_L), r"\mu m")

$$ u_c^2(E,L)=2.08569916666667 \cdot 10^{-11} L^{2} + 667.60847769555\ \mathrm{ \mu m^2}$$

$$ u_c(E,L)=\sqrt{2.08569916666667 \cdot 10^{-11} L^{2} + 667.60847769555}\ \mathrm{ \mu m}$$

$$ U(E,L)=\sqrt{8.34279666666666 \cdot 10^{-11} L^{2} + 2670.4339107822}\ \mathrm{ \mu m}$$

Incertidumbre expandida

In [48]:
u2cE_expandida = 4 * u2cE
u2cE = u2cE_expandida

print("(en um^2):")
u2cE

(en um^2):


8.34279666666666e-11*L**2 + 2670.4339107822

Sacando la raiz cuadrada

In [49]:
u2cE = u2cE**(1/2)
u2cE


(8.34279666666666e-11*L**2 + 2670.4339107822)**0.5

Finalmente redondeando el valor de $u^2(E)
$

Para una longitud $L$ en $\mu m $ es

In [50]:

# Forma cerrada final a partir de la varianza expandida (u2cE_expandida)
a_L = float(u2cE_expandida.coeff(L, 2))
b_L = float(u2cE_expandida.coeff(L, 0))
display(Markdown(
    rf"$U(E,L) = \sqrt{{{a_L:.3e}\,L^2 + {b_L:.1f}}}\ \mu m$, con $L$ en mm"
))

# Comprobacion en los puntos medidos
for Lv in VP:
    print(f"L = {Lv:6.1f} mm -> U = {np.sqrt(a_L*Lv**2 + b_L):.4f} um")

L =   50.0 mm -> U = 51.6762 um
L =  100.0 mm -> U = 51.6762 um
L =  150.0 mm -> U = 51.6762 um
L =  200.0 mm -> U = 51.6762 um
L =  250.0 mm -> U = 51.6762 um


$U(E,L) = \sqrt{8.343e-11\,L^2 + 2670.4}\ \mu m$, con $L$ en mm

Redondeando

In [51]:
# Balance de incertidumbres calculado en la celda anterior
# ("Balance de componentes"), que define comp, u_Li, u_cE y U_E.

# NOTACION: sigma (incertidumbre tipica) equivale a u en la GUM.
for nombre, valor in comp.items():
    print(f"{nombre:32s} {valor:6.2f} um  ({100*valor**2/u_Li**2:5.1f} %)")
print(f"{chr(963)}(Li){'':25s} {u_Li:6.2f} um")
print(f"{chr(963)}(LBP){'':24s} {u_BP:6.4f} um")
print(f"{chr(963)}_c(E){'':24s} {u_cE:6.2f} um")
print(f"{'U(E) = 2' + chr(963) + '_c(E)':32s} {U_E:6.2f} um")

export_table(
    TAB_DIR / "t6.tex",
    # NOTACION: se usa sigma (incertidumbre tipica) para ser coherente con
    # el marco teorico del capitulo 1 y con las secciones 2.1.4-2.1.7.
    # NOTA: si mas adelante se prefiere la notacion GUM con "u", basta con
    # descomentar las lineas marcadas [GUM] y comentar las [SIGMA].
    header=[
        r"Contribución",
        r"$\sigma_i$ ($\mu$m)",        # [SIGMA]  notacion GUM: r"$u_i$ ($\mu$m)"
        r"Peso en la varianza (\%)",
    ],
    rows=[
        [nombre, fmt(valor, 2), fmt(100 * valor ** 2 / u_Li ** 2, 1)]
        for nombre, valor in comp.items()
    ]
    + [
        # [SIGMA] notacion GUM: r"$u(L_i)$"
        [r"$\sigma(L_i)$", fmt(u_Li, 2), "100.0"],
        # [SIGMA] notacion GUM: r"$u(L_{BP})$"
        [r"$\sigma(L_{BP})$", f"{u_BP:.4f}", f"{100*u_BP**2/u_cE**2:.3f}"],
        # [SIGMA] notacion GUM: r"$u_c(E) = \sqrt{u^2(L_i) + u^2(L_{BP})}$"
        [r"$\sigma_c(E) = \sqrt{\sigma^2(L_i) + \sigma^2(L_{BP})}$", fmt(u_cE, 2), r"—"],
        # [SIGMA] notacion GUM: r"$U(E) = 2\,u_c(E)$", fmt(U_E, 2)
        [r"$U(E) = 2\,\sigma_c(E)$", fmt(U_E, 2), r"—"],
    ],
    caption=(
        "Balance de incertidumbres de la indicación del pie de rey y "
        "contribuciones a la varianza combinada."
    ),
    label="tab:pie-rey-balance",
    colspec="lrr",
)

Contacto parcial $E$              14.43 um  ( 31.2 %)
Repetibilidad $R$                 12.25 um  ( 22.5 %)
Cambio de escala $S$               9.62 um  ( 13.9 %)
Contacto lineal $L$                2.89 um  (  1.2 %)
Superficie completa $J$            0.00 um  (  0.0 %)
Cruce de interiores $K$            0.00 um  (  0.0 %)
Resolución $r$                    14.43 um  ( 31.2 %)
σ(Li)                           25.84 um
σ(LBP)                         0.1260 um
σ_c(E)                          25.84 um
U(E) = 2σ_c(E)                    51.68 um
escrito: inf/c3-calibration/pie-de-rey/tablas/t6.tex
